**Purpose:** FINAL Deep-RL PPO portfolio agent (technical features only).

**Inputs:** `03_portfolio/dataset.parquet`

**Outputs:** `DRLPPO_technical_weights.npy`, `{prefix}_cum.png`, `{prefix}_daily_returns.png`, `{prefix}_weights.png`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.config import PROJECT_ROOT
from src.seeds import SEED_PORTFOLIO_DRL_PPO_V9_1_3


### DRL PPO

- Architecture is based on [0df79], namely the lookback, hyperparameter tunning, model used, reward function, and so on.

- Hyperparameters were selected using an expanding-window walk-forward validation scheme. Three folds were constructed: training on 5, 6, and 7 years of historical data respectively, each followed by a 1-year validation period. The final model was trained on the full 8-year pre-test period and evaluated on a strictly out-of-sample 2-year test set (2023–2025). This preserves temporal order and avoids information leakage common in random cross-validation for time-series data (Hyndman & Athanasopoulos). Some sources: [
0df79,
[00001](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html?utm_source=chatgpt.com),
[00002](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3104847)
]

- Features come from the buckets (dataset.ipynb): one per bucket, OHLC yes or no, VIXIndex yes or no, and Close is mandatory.

---

- Most machine learning studies using technical indicators retain the indicators in their standard definitions and apply a single global normalization step such as min–max or z-score scaling. In contrast, this work applies simple feature-specific transformations based on the statistical properties of each indicator. Price levels are converted to log returns, while price-related indicators such as Open, High, Low, and moving averages are expressed as log ratios relative to the closing price to remove price-level effects. Bounded oscillators such as RSI and ADX are rescaled to a symmetric interval around zero, heavy-tailed volume-related variables are compressed using a logarithmic transformation, and the remaining continuous features are standardized using robust statistics based on the median and median absolute deviation (MAD). These transformations improve comparability across assets, reduce the influence of extreme observations, and provide numerically stable inputs for the reinforcement learning model while preserving the economic interpretation of the indicators.

---

To improve the stability and learning efficiency of the reinforcement learning model, several transformations were applied to the input features before training. Financial time series often contain large differences in scale, non-stationary price levels, and extreme observations. These characteristics can make learning difficult for neural networks. Therefore, the transformations were designed to remove price-level effects, normalize feature scales, and reduce the impact of outliers. All transformation parameters were estimated using only the training portion of each walk-forward split and then applied to the validation period. This ensures that the model does not use information from the future and prevents look-ahead bias.

Raw price levels were not used directly because they are non-stationary and differ greatly across assets. Instead, prices were converted into relative measures. Closing prices were transformed into one-day log returns, which represent daily percentage-like price changes. The Open, High, and Low prices were expressed relative to the closing price using logarithmic ratios. Similarly, moving average indicators such as simple moving averages were transformed into the logarithmic ratio between the close price and the moving average. These transformations remove the absolute price level and instead capture relative price movements, which are more stable over time and more comparable across assets.

Technical indicators such as MACD, CCI, Bollinger Band values, and other momentum or volatility indicators were normalized using a robust scaling approach. The values were first slightly clipped to limit the influence of extreme observations and were then standardized using statistics based on the median and the median absolute deviation. This method was chosen because financial data frequently contain outliers, and robust statistics are less sensitive to extreme values than traditional normalization based on the mean and standard deviation. As a result, the scaled features provide more stable inputs for the learning algorithm.

Indicators that naturally lie within a fixed range, such as the Relative Strength Index (RSI) and the Average Directional Index (ADX), were treated differently. Since these indicators typically take values between 0 and 100, they were simply rescaled to approximately the range between −1 and 1. This preserves the relative interpretation of the indicator while placing it on a scale that is easier for the neural network to process.

Volume-related variables, such as trading volume, On-Balance Volume (OBV), and the Accumulation/Distribution Indicator (ADI), can have very large magnitudes and highly skewed distributions. To address this, a logarithmic transformation was applied before normalization. This compresses very large values while preserving the sign and direction of the signal, making the distribution more suitable for learning.

Finally, to ensure numerical stability during training, feature values were clipped within reasonable bounds after normalization. Financial time series often contain rare but extremely large observations, which can destabilize neural network training. Limiting these extreme values helps maintain stable inputs while still preserving the general structure of the data.

When included, the market indicator used in the model, the VIX index, was also transformed. Instead of using its raw level, it was converted into daily log returns and normalized using the same robust scaling procedure as other unbounded indicators. This allows the model to focus on changes in market volatility rather than the absolute level of the index.

Overall, these transformations were implemented to provide the reinforcement learning model with stable and comparable input signals. By removing price-level effects, normalizing feature scales, and limiting the influence of outliers, the transformed features are better suited for training the PPO-based portfolio allocation model.

---

"A single environment was used due to the sequential nature of financial data; parallelism would require care to avoid temporal overlap between environments, and was left for future work." ; The only way parallelism would make sense in your setup is if you parallelized both Optuna trials and final training consistently — but that would massively increase compute cost and complexity for a thesis, with questionable benefit given the financial data constraints I mentioned. ; "Parallel environments were intentionally avoided to ensure consistency between the hyperparameter search and the final training regime."

---

changed patient in middle fo traisl to 2 and 3 so its faster lol

---

# DRL PPO Portfolio Allocation

This notebook has two main stages:

1. **Hyperparameter Tuning** (`run_optuna_search`) — expanding-window walk-forward validation with Optuna (TPE sampler). Three folds train on 5 / 6 / 7 years and validate on the following 1-year window.
2. **Final Run** (`run_final_train_test`) — trains on the full 8-year pre-test period and evaluates on a 2-year out-of-sample test set with an optional mid-test refit.

Key design decisions:
- Features are selected per-bucket (one per bucket, OHLC on/off, VIX on/off).  
- Stationary per-feature transforms are fit on the training window only (no look-ahead).  
- Reward: warm-up on net returns → Differential Sharpe Ratio (DSR), clipped.  
- Costs use fixed bid-ask spreads (cost is cost — not a tunable parameter).


---
## 1 · Imports & Shared Infrastructure

All classes and helpers shared between the tuning and final-run stages live here:
`FeatureTransformSpec`, `FittedTransform`, `OneDayHoldPortfolioEnv`,
`EarlyStopSharpeCallback`, `DiagnosticsCallback`, `PerAssetFeaturesExtractor`,
and the various utility functions.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Core imports
# ─────────────────────────────────────────────────────────────────────────────

from __future__ import annotations

import os
import json
import time
import math
import random
import logging
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

import gymnasium as gym
from gymnasium import spaces

import torch as th
from torch import nn

import optuna
from optuna.samplers import TPESampler

from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.logger import configure as sb3_configure_logger
from stable_baselines3.common.callbacks import BaseCallback, CallbackList

import matplotlib.pyplot as plt


---
## 2 · Logging Helper


In [ ]:
def setup_logger(log_dir: str, name_prefix: str = "ppo") -> logging.Logger:
    #os.makedirs(log_dir, exist_ok=True)
    logger = logging.getLogger(f"{name_prefix}_{os.path.basename(log_dir)}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    fmt = logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s")

    ch = logging.StreamHandler()
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    #fh = logging.FileHandler(os.path.join(log_dir, "run.log"))
    #fh.setFormatter(fmt)
    #logger.addHandler(fh)

    logger.propagate = False
    return logger


---
## 2b · Per-Asset Feature Extractor (Structured Observation)

Instead of a flat MLP over the entire concatenated observation, we use a
**permutation-equivariant per-asset encoder**: a small shared MLP processes
each asset's feature window independently, then the asset embeddings are
max-pooled and concatenated with the market signal and availability mask
before being fed to the actor/critic heads.

This preserves the `[T, N, F]` structure of the observation space that was
lost by flattening, and halves the number of parameters in the encoder
(weight-sharing across assets).

The encoder width is controlled by `encoder_width` (tuned by Optuna) and
`net_arch` governs the shared MLP head after pooling (same as before).


In [ ]:
class PerAssetFeaturesExtractor(BaseFeaturesExtractor):
    """
    Structured feature extractor that respects the per-asset layout of the
    observation vector produced by OneDayHoldPortfolioEnv.

    Observation layout (flat vector):
        [lookback * N * F]  per-asset feature windows
        [1]                 market signal (optional)
        [N]                 previous portfolio weights
        [N]                 asset availability mask

    Processing:
        1. Reshape per-asset block to (N, lookback * F)
        2. Pass each asset through a shared linear encoder -> (N, encoder_width)
        3. max-pool across assets -> (encoder_width,)
        4. Concatenate with market + prev_weights + avail_mask
        5. Output fed to the actor/critic MLP head (net_arch)
    """

    def __init__(
        self,
        observation_space,
        n_assets:       int,
        lookback:       int,
        n_features:     int,
        has_market:     bool,
        encoder_width:  int  = 64,
    ):
        self.n_assets      = n_assets
        self.lookback      = lookback
        self.n_features    = n_features
        self.has_market    = has_market
        self.encoder_width = encoder_width

        # Extra dims after the per-asset block
        n_extra = (1 if has_market else 0) + n_assets + n_assets  # market + prev_w + avail
        features_dim = encoder_width + n_extra

        super().__init__(observation_space, features_dim=features_dim)

        asset_input_dim = lookback * n_features
        # Shared per-asset encoder (weight-tied across all assets)
        self.asset_encoder = nn.Sequential(
            nn.Linear(asset_input_dim, encoder_width),
            nn.Tanh(),
        )

    def forward(self, obs: th.Tensor) -> th.Tensor:
        B = obs.shape[0]
        N, L, F = self.n_assets, self.lookback, self.n_features

        # Split observation into its logical components
        asset_block_len = N * L * F
        asset_flat = obs[:, :asset_block_len]                        # (B, N*L*F)
        rest        = obs[:, asset_block_len:]                       # (B, n_extra)

        # Reshape to (B, N, L*F) and apply shared encoder per asset
        asset_flat = asset_flat.view(B, N, L * F)                   # (B, N, L*F)
        asset_emb  = self.asset_encoder(asset_flat)                  # (B, N, encoder_width)

        # Max-pool across assets.
        # For heterogeneous sector ETFs, max-pool preserves the strongest
        # per-feature activation without averaging it away, which is more
        # informative than mean-pool when assets differ structurally.
        pooled = asset_emb.max(dim=1).values                        # (B, encoder_width)

        return th.cat([pooled, rest], dim=1)                         # (B, features_dim)


---
## 3 · Walk-Forward & Train/Test Splits

- **Tuning splits** — 3 expanding folds: train 5 / 6 / 7 years, validate 1 year each.  
- **Final split** — train 2015-07-01 → 2023-06-30, test 2023-07-01 → 2025-06-30.  
- **Mid-test refit split** — the test window is divided at a configurable mid-point (default 2024-07-01) to allow a model refit with all data up to that date before trading the second half.


In [ ]:
def make_walk_forward_splits(dates_index: pd.DatetimeIndex):
    """Three expanding-window folds for Optuna tuning."""
    splits = []
    periods = [
        (pd.Timestamp("2015-07-01"), pd.Timestamp("2020-06-30"),
         pd.Timestamp("2020-07-01"), pd.Timestamp("2021-06-30")),
        (pd.Timestamp("2015-07-01"), pd.Timestamp("2021-06-30"),
         pd.Timestamp("2021-07-01"), pd.Timestamp("2022-06-30")),
        (pd.Timestamp("2015-07-01"), pd.Timestamp("2022-06-30"),
         pd.Timestamp("2022-07-01"), pd.Timestamp("2023-06-30")),
    ]
    for tr_start, tr_end, v_start, v_end in periods:
        train_idx = np.where((dates_index >= tr_start) & (dates_index <= tr_end))[0]
        val_idx   = np.where((dates_index >= v_start)  & (dates_index <= v_end))[0]
        if len(train_idx) == 0 or len(val_idx) == 0:
            print(f"[SKIP SPLIT] train {tr_start.date()}-{tr_end.date()} "
                  f"val {v_start.date()}-{v_end.date()} "
                  f"| lens train={len(train_idx)} val={len(val_idx)}")
            continue
        splits.append((train_idx, val_idx))
    if not splits:
        raise ValueError("No valid walk-forward splits. Check dataset date coverage.")
    return splits


def make_final_train_test_split(
    dates_index: pd.DatetimeIndex,
    train_start: str = "2015-07-01",
    train_end:   str = "2023-06-30",
    test_start:  str = "2023-07-01",
    test_end:    str = "2025-06-30",
) -> Tuple[np.ndarray, np.ndarray]:
    """Single final train / test split."""
    ts = pd.Timestamp(train_start); te = pd.Timestamp(train_end)
    ss = pd.Timestamp(test_start);  se = pd.Timestamp(test_end)
    train_idx = np.where((dates_index >= ts) & (dates_index <= te))[0]
    test_idx  = np.where((dates_index >= ss) & (dates_index <= se))[0]
    if len(train_idx) == 0 or len(test_idx) == 0:
        raise ValueError(
            f"Invalid final split. train={len(train_idx)} test={len(test_idx)}. "
            f"Check dataset date coverage."
        )
    return train_idx, test_idx


def make_midtest_refit_split(
    test_idx: np.ndarray,
    dates_index: pd.DatetimeIndex,
    mid_date: str = "2024-07-01",
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Split the test window into two halves around mid_date:
      test_first  : 2023-07-01 .. mid_date - 1 day  (model trained up to 2023-06-30)
      test_second : mid_date   .. 2025-06-30         (model refitted up to mid_date - 1 day)
    """
    mid = pd.Timestamp(mid_date)
    test_dates = dates_index[test_idx]
    first_mask  = test_dates < mid
    second_mask = test_dates >= mid
    test_first  = test_idx[first_mask]
    test_second = test_idx[second_mask]
    if len(test_first) == 0 or len(test_second) == 0:
        raise ValueError(
            f"Mid-test split at {mid_date} produced an empty half. "
            f"first={len(test_first)} second={len(test_second)}."
        )
    return test_first, test_second


def split_train_for_early_stop(
    train_idx: np.ndarray,
    dates_index: pd.DatetimeIndex,
    es_years: int = 1,
    min_train_days: int = 252,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Reserve the last `es_years` calendar year(s) of the training window for
    early-stopping validation, keeping the rest as the core training set.
    """
    train_dates = dates_index[train_idx]
    train_end = train_dates[-1]
    es_start_date = train_end - pd.DateOffset(years=es_years) + pd.Timedelta(days=1)

    es_mask   = train_dates >= es_start_date
    core_mask = ~es_mask
    core_idx  = train_idx[core_mask]
    es_idx    = train_idx[es_mask]

    # Fallback if calendar split is too small
    if len(core_idx) < min_train_days or len(es_idx) < min_train_days // 2:
        cut = max(min_train_days, len(train_idx) - 252 * es_years)
        cut = min(cut, len(train_idx) - 1)
        core_idx = train_idx[:cut]
        es_idx   = train_idx[cut:]

    if len(core_idx) == 0 or len(es_idx) == 0:
        raise ValueError("Failed to create non-empty early-stopping split inside training window.")

    return core_idx, es_idx


---
## 4 · Feature Selection

Features are drawn from five buckets. Optuna chooses one indicator per bucket (Trend, Momentum, Volume, Volatility) and can optionally include OHLC and the VIX market feature.  
`BBands` is treated as a single combined choice that expands to `[BBLower, BBUpper]`.


In [ ]:
def pick_features_from_buckets(trial: optuna.Trial, features: Dict[str, List[str]]) -> Dict[str, Any]:
    """
    - Price    : always log_ret_1d
    - OHLC     : on/off as group
    - Trend / Momentum / Volume : pick one each
    - Volatility : pick one (or BBands = BBLower + BBUpper combined)
    - Market   : on/off (VIXIndexClose)
    """
    picks: Dict[str, Any] = {}
    picks["Price"] = "log_ret_1d"

    use_ohlc = trial.suggest_categorical("use_ohlc", [0, 1])
    picks["OHLC"] = ["Open", "High", "Low"] if use_ohlc == 1 else []

    for bucket in ["Trend", "Momentum", "Volume"]:
        picks[bucket] = trial.suggest_categorical(f"pick_{bucket}", features[bucket])

    vol_options = [f for f in features["Volatility"] if f not in ("BBLower", "BBUpper")]
    vol_options.append("BBands")
    vol_choice = trial.suggest_categorical("pick_Volatility", vol_options)
    picks["Volatility"] = ["BBLower", "BBUpper"] if vol_choice == "BBands" else vol_choice

    use_market = trial.suggest_categorical("use_market", [0, 1])
    picks["Market"] = "VIXIndexClose" if use_market == 1 else None

    return picks


def get_per_asset_feature_list(picks: Dict[str, Any]) -> List[str]:
    """Ordered, deduplicated list of per-asset feature names (log_ret_1d first)."""
    feats: List[str] = ["log_ret_1d"]
    for f in picks.get("OHLC", []):
        feats.append(f)
    for bucket in ["Trend", "Momentum", "Volume", "Volatility"]:
        val = picks[bucket]
        if isinstance(val, list):
            feats.extend(val)
        else:
            feats.append(val)
    seen: set = set()
    out: List[str] = []
    for f in feats:
        if f not in seen:
            out.append(f)
            seen.add(f)
    return out


def required_df_columns(assets: List[str], per_asset_feats: List[str], picks: Dict[str, Any]) -> List[str]:
    cols: List[str] = []
    for a in assets:
        cols.append(f"{a}_Close")
        for f in per_asset_feats:
            if f == "log_ret_1d":
                continue
            cols.append(f"{a}_{f}")
    if picks.get("Market") is not None:
        cols.append(picks["Market"])
    return cols


---
## 5 · Availability Mask

An asset is tradable at step *t* only if:
- `close[t]` and `close[t+1]` are both finite (needed to compute the 1-day return), and
- every feature over the lookback window `[t−lookback+1, …, t]` is finite.


In [ ]:
def compute_availability_mask(
    closes_raw: np.ndarray,   # [T, N]
    feats_raw:  np.ndarray,   # [T, N, F]
    lookback:   int,
) -> np.ndarray:
    T, N = closes_raw.shape
    avail = np.zeros((T, N), dtype=bool)

    finite_close_t  = np.isfinite(closes_raw)
    finite_close_t1 = np.isfinite(
        np.vstack([closes_raw[1:], np.full((1, N), np.nan, dtype=np.float32)])
    )
    finite_feats = np.isfinite(feats_raw).all(axis=2)   # [T, N]

    for t in range(lookback - 1, T - 1):
        win_ok = finite_feats[t - lookback + 1 : t + 1].all(axis=0)
        avail[t] = finite_close_t[t] & finite_close_t1[t] & win_ok

    avail[-1, :] = False
    return avail


---
## 6 · Stationary Feature Transforms

All transform parameters (medians, MADs, winsorisation bounds) are **fit on the training split only** and then applied to both the training and out-of-sample windows, preventing any look-ahead bias.

| Feature type | Transform |
|---|---|
| `log_ret_1d` | Robust z-score (winsorise → median/MAD) |
| `RSI`, `ADX` | Bounded 0–100 → rescaled to ≈ [−1, 1] |
| `Volume`, `OBV`, `ADI`, `VolumeVariation` | Signed log1p then robust z-score |
| `Open`, `High`, `Low` | Log-ratio relative to Close, then robust z-score |
| `SMA*` | Log-ratio Close/SMA, then robust z-score |
| Everything else | Robust z-score |

After normalisation all values are clipped to `±clip_value`.


In [ ]:
def _winsorize_fit(x: np.ndarray, q: float) -> Tuple[float, float]:
    lo = np.nanquantile(x, q)
    hi = np.nanquantile(x, 1.0 - q)
    if not np.isfinite(lo): lo = 0.0
    if not np.isfinite(hi): hi = 0.0
    if hi < lo: hi = lo
    return float(lo), float(hi)

def _winsorize_apply(x: np.ndarray, lo: float, hi: float) -> np.ndarray:
    return np.clip(x, lo, hi)


@dataclass
class FeatureTransformSpec:
    kind:        str    # "robust_z" | "bounded_0_100" | "log1p_robust_z" | "identity" | "clip"
    winsor_q:    float = 0.01
    clip_value:  float = 10.0


@dataclass
class FittedTransform:
    spec:   FeatureTransformSpec
    lo:     float = 0.0
    hi:     float = 0.0
    center: float = 0.0
    scale:  float = 1.0

    def transform(self, x: np.ndarray) -> np.ndarray:
        x = x.astype(np.float32)
        x = np.nan_to_num(x, nan=np.nan, posinf=np.nan, neginf=np.nan)

        if self.spec.kind in ("robust_z", "log1p_robust_z"):
            x = _winsorize_apply(x, self.lo, self.hi)

        if self.spec.kind == "identity":
            y = x
        elif self.spec.kind == "bounded_0_100":
            y = (x - 50.0) / 50.0
        elif self.spec.kind == "log1p_robust_z":
            y = np.sign(x) * np.log1p(np.abs(x))
            y = (y - self.center) / self.scale
        elif self.spec.kind == "robust_z":
            y = (x - self.center) / self.scale
        elif self.spec.kind == "clip":
            y = np.clip(x, -self.spec.clip_value, self.spec.clip_value)
        else:
            raise ValueError(f"Unknown transform kind: {self.spec.kind}")

        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
        if self.spec.clip_value is not None:
            y = np.clip(y, -self.spec.clip_value, self.spec.clip_value)
        return y.astype(np.float32)


def default_transform_spec(feature_name: str) -> FeatureTransformSpec:
    """Choose the appropriate transformation by feature semantics."""
    name = feature_name.lower()
    if feature_name == "log_ret_1d":
        return FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)
    if name in ("rsi", "adx"):
        return FeatureTransformSpec(kind="bounded_0_100", clip_value=2.0)
    if name in ("volume", "volumevariation", "obv", "adi"):
        return FeatureTransformSpec(kind="log1p_robust_z", winsor_q=0.01, clip_value=8.0)
    if name in ("open", "high", "low"):
        return FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)
    return FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)


def fit_transforms_on_train(
    X_train_raw:    np.ndarray,           # [T, N, F]
    feat_names:     List[str],
    market_train_raw: Optional[np.ndarray],  # [T, 1] or None
) -> Tuple[List[FittedTransform], Optional[FittedTransform]]:
    """Fit per-feature transforms by pooling over time and assets."""
    fitted: List[FittedTransform] = []
    for j, fname in enumerate(feat_names):
        spec = default_transform_spec(fname)
        vals = X_train_raw[:, :, j].reshape(-1)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            fitted.append(FittedTransform(spec=spec)); continue

        if spec.kind in ("robust_z", "log1p_robust_z"):
            lo, hi = _winsorize_fit(vals, spec.winsor_q)
            vals_w = _winsorize_apply(vals, lo, hi)
            if spec.kind == "log1p_robust_z":
                vals_w = np.sign(vals_w) * np.log1p(np.abs(vals_w))
            center = float(np.nanmedian(vals_w))
            mad    = float(np.nanmedian(np.abs(vals_w - center)))
            scale  = 1.4826 * mad
            if not np.isfinite(scale) or scale < 1e-6:
                scale = float(np.nanstd(vals_w))
            if not np.isfinite(scale) or scale < 1e-6:
                scale = 1.0
            fitted.append(FittedTransform(spec=spec, lo=lo, hi=hi, center=center, scale=scale))
        elif spec.kind in ("bounded_0_100", "identity", "clip"):
            fitted.append(FittedTransform(spec=spec))
        else:
            raise ValueError(spec.kind)

    market_ft: Optional[FittedTransform] = None
    if market_train_raw is not None:
        spec = FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)
        v = market_train_raw[:, 0]
        v = v[np.isfinite(v)]
        lr = np.log(v[1:] / v[:-1]) if v.size > 2 else np.array([0.0], dtype=np.float32)
        lr = lr[np.isfinite(lr)]
        lo, hi    = _winsorize_fit(lr, spec.winsor_q)
        lr_w      = _winsorize_apply(lr, lo, hi)
        center    = float(np.nanmedian(lr_w))
        mad       = float(np.nanmedian(np.abs(lr_w - center)))
        scale     = 1.4826 * mad
        if not np.isfinite(scale) or scale < 1e-6:
            scale = float(np.nanstd(lr_w))
        if not np.isfinite(scale) or scale < 1e-6:
            scale = 1.0
        market_ft = FittedTransform(spec=spec, lo=lo, hi=hi, center=center, scale=scale)

    return fitted, market_ft


def apply_transforms(
    X_raw:      np.ndarray,
    fitted:     List[FittedTransform],
    market_raw: Optional[np.ndarray],
    market_ft:  Optional[FittedTransform],
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    X = np.empty_like(X_raw, dtype=np.float32)
    for j, ft in enumerate(fitted):
        X[:, :, j] = ft.transform(X_raw[:, :, j])

    m_out: Optional[np.ndarray] = None
    if market_raw is not None and market_ft is not None:
        v   = market_raw[:, 0].astype(np.float32)
        lr  = np.full_like(v, np.nan, dtype=np.float32)
        lr[1:] = np.log(v[1:] / v[:-1])
        lr  = np.nan_to_num(lr, nan=0.0, posinf=0.0, neginf=0.0)
        m_out = market_ft.transform(lr).reshape(-1, 1).astype(np.float32)

    return X, m_out


---
## 7 · Build Raw Arrays from DataFrame


In [ ]:
def build_raw_arrays(
    df:       pd.DataFrame,
    assets:   List[str],
    picks:    Dict[str, Any],
    features: Dict[str, List[str]],
    logger:   logging.Logger,
) -> Tuple[np.ndarray, np.ndarray, List[str], Optional[np.ndarray]]:
    """
    Returns:
        closes  : [T, N]  raw close prices
        X_raw   : [T, N, F]  raw (pre-transform) feature array
        feat_names : list of per-asset feature names (length F)
        market  : [T, 1] raw VIX close, or None
    """
    per_asset_feats = get_per_asset_feature_list(picks)
    cols = required_df_columns(assets, per_asset_feats, picks)
    missing = [c for c in cols if c not in df.columns]
    if missing:
        logger.error(f"Missing columns ({len(missing)}): {missing[:25]}{'...' if len(missing) > 25 else ''}")
        raise KeyError("Dataset missing required columns for this feature set.")

    T, N, F = len(df), len(assets), len(per_asset_feats)
    closes = np.zeros((T, N), dtype=np.float32)
    X_raw  = np.zeros((T, N, F), dtype=np.float32)

    for i, a in enumerate(assets):
        c = df[f"{a}_Close"].to_numpy(np.float32)
        closes[:, i] = c

        for j, f in enumerate(per_asset_feats):
            if f == "log_ret_1d":
                lr = np.full_like(c, np.nan, dtype=np.float32)
                lr[1:] = np.log(c[1:] / c[:-1])
                X_raw[:, i, j] = lr
            elif f in ("Open", "High", "Low"):
                raw   = df[f"{a}_{f}"].to_numpy(np.float32)
                rel   = np.full_like(raw, np.nan, dtype=np.float32)
                valid = (raw > 0) & (c > 0) & np.isfinite(raw) & np.isfinite(c)
                rel[valid] = np.log(raw[valid] / c[valid])
                X_raw[:, i, j] = rel
            elif f.startswith("SMA"):
                sma   = df[f"{a}_{f}"].to_numpy(np.float32)
                rel   = np.full_like(sma, np.nan, dtype=np.float32)
                valid = (sma > 0) & (c > 0) & np.isfinite(sma) & np.isfinite(c)
                rel[valid] = np.log(c[valid] / sma[valid])
                X_raw[:, i, j] = rel
            else:
                X_raw[:, i, j] = df[f"{a}_{f}"].to_numpy(np.float32)

    market: Optional[np.ndarray] = None
    if picks.get("Market") is not None:
        market = df[picks["Market"]].to_numpy(np.float32).reshape(-1, 1)

    return closes, X_raw, per_asset_feats, market


---
## 8 · Differential Sharpe Ratio (DSR) Reward

The reward transitions from a simple net-return warm-up phase to the online Differential Sharpe Ratio (Moody & Saffell, 1998). DSR provides a risk-adjusted reward that adapts online without needing a fixed window.


In [ ]:
@dataclass
class DiffSharpeState:
    A:    float = 0.0
    B:    float = 0.0
    init: bool  = True


def diff_sharpe_update(state: DiffSharpeState, r: float, eta: float = 0.01, eps: float = 1e-8) -> float:
    if state.init:
        state.A    = r
        state.B    = r * r
        state.init = False
        return 0.0

    A_prev    = state.A
    B_prev    = state.B
    var_prev  = max(B_prev - A_prev * A_prev, 0.0)

    num = (B_prev * (r - A_prev)) - 0.5 * A_prev * ((r * r) - B_prev)
    den = (var_prev ** 1.5) + eps
    dsr = float(num / den)

    state.A = A_prev + eta * (r - A_prev)
    state.B = B_prev + eta * ((r * r) - B_prev)

    return 0.0 if not np.isfinite(dsr) else dsr


---
## 9 · Portfolio Environment

`OneDayHoldPortfolioEnv` implements the Gymnasium interface.  
The agent outputs raw logits for each asset; a **masked softmax** converts them to a valid long-only allocation (unavailable assets receive zero weight). The reward is:

$$r_t = \text{scale} \times \begin{cases} R_t^{\text{net}} & \text{warm-up} \\ \text{clip}(\Delta S_t) & \text{otherwise} \end{cases}$$

where $R_t^{\text{net}} = w_t^\top \frac{p_{t+1} - p_t}{p_t} - \text{cost}_t$ and costs use fixed bid-ask spreads only.


In [ ]:
class OneDayHoldPortfolioEnv(gym.Env):
    """
    One-day hold portfolio environment.
    Action  -> masked-softmax weights at t
    Reward  -> linearly blended from reward_scale_net*net_return (step 0)
               to reward_scale_dsr*clip(DSR(net_return)) (step dsr_warmup_steps)
    Costs use fixed bid-ask spreads (not tunable).
    """
    metadata = {"render_modes": []}

    def __init__(
        self,
        closes_raw:        np.ndarray,
        feats:             np.ndarray,
        avail_mask:        np.ndarray,
        spreads:           np.ndarray,
        market:            Optional[np.ndarray] = None,
        dsr_eta:           float = 0.01,
        dsr_clip:          float = 10.0,
        dsr_warmup_steps:  int   = 252,
        reward_scale_net:  float = 100.0,
        reward_scale_dsr:  float = 10.0,
        start_idx:         int   = 0,
        end_idx:           Optional[int] = None,
        lookback:          int   = 22,
    ):
        super().__init__()
        self.closes  = closes_raw.astype(np.float32)
        self.feats   = feats.astype(np.float32)
        self.avail   = avail_mask
        self.market  = market.astype(np.float32) if market is not None else None
        self.spreads = spreads.astype(np.float32)

        self.dsr_eta          = float(dsr_eta)
        self.dsr_clip         = float(dsr_clip)
        self.dsr_warmup_steps = int(dsr_warmup_steps)
        self.reward_scale_net = float(reward_scale_net)
        self.reward_scale_dsr = float(reward_scale_dsr)

        self.N        = self.closes.shape[1]
        self.F        = self.feats.shape[2]
        self.T        = self.closes.shape[0]
        self.start_idx = int(start_idx)
        self.end_idx   = int(end_idx) if end_idx is not None else (self.T - 2)
        self.lookback  = int(lookback)

        self.action_space = spaces.Box(low=-5.0, high=5.0, shape=(self.N,), dtype=np.float32)
        obs_dim = self.lookback * self.N * self.F + self.N + self.N
        if self.market is not None:
            obs_dim += 1
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)

        self._t = self._w_prev = self._dsr = self._step = None

    @staticmethod
    def _masked_softmax(a: np.ndarray, mask: np.ndarray) -> np.ndarray:
        a = a.astype(np.float32)
        if not mask.any():
            return np.zeros_like(a, dtype=np.float32)
        logits = a.copy(); logits[~mask] = -1e9
        logits -= np.max(logits[mask])
        e = np.zeros_like(logits, dtype=np.float32)
        e[mask] = np.exp(logits[mask])
        s = float(np.sum(e[mask]))
        w = np.zeros_like(e, dtype=np.float32)
        w[mask] = e[mask] / max(s, 1e-12)
        return w

    @staticmethod
    def _entropy(w: np.ndarray) -> float:
        w = np.asarray(w, dtype=np.float64)
        return float(-np.sum(w * np.log(w + 1e-12)))

    def _get_obs(self) -> np.ndarray:
        x = self.feats[self._t - self.lookback + 1 : self._t + 1].reshape(-1)
        parts = [x]
        if self.market is not None:
            parts.append(self.market[self._t].reshape(-1))
        parts.append(self._w_prev.astype(np.float32))
        parts.append(self.avail[self._t].astype(np.float32))
        return np.nan_to_num(np.concatenate(parts).astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        self._t      = max(self.start_idx, self.lookback - 1)
        self._w_prev = np.ones(self.N, dtype=np.float32) / self.N
        self._dsr    = DiffSharpeState()
        self._step   = 0
        return self._get_obs(), {}

    def step(self, action: np.ndarray):
        t = self._t
        if t >= self.end_idx:
            w  = self._w_prev if self._w_prev is not None else np.ones(self.N, dtype=np.float32) / self.N
            ew = 1.0 / self.N
            info = {
                "net_ret": 0.0, "gross_ret": 0.0, "cost": 0.0, "turnover": 0.0,
                "dsr": 0.0, "w": w.astype(np.float32),
                "l1_to_ew": float(np.sum(np.abs(w - ew))),
                "went": self._entropy(w), "wmax": float(np.max(w)),
                "tradable_n": int(self.avail[t].sum()) if t < self.avail.shape[0] else 0,
            }
            return self._get_obs(), 0.0, True, False, info

        mask  = self.avail[t]
        w_new = self._masked_softmax(action, mask)

        p0  = np.where(self.closes[t] <= 0, np.nan, self.closes[t])
        rel = np.where(np.isfinite(self.closes[t + 1] / p0 - 1.0),
                       self.closes[t + 1] / p0 - 1.0, 0.0).astype(np.float32)

        gross    = float(np.dot(w_new, rel))
        dw       = w_new - self._w_prev
        turnover = float(np.sum(np.abs(dw)))
        cost     = float(0.5 * np.sum(np.abs(dw) * self.spreads))
        net      = gross - cost

        dsr_val  = float(np.clip(diff_sharpe_update(self._dsr, net, eta=self.dsr_eta),
                                 -self.dsr_clip, self.dsr_clip))

        # Linear blend: ramp from pure net-return reward to pure DSR over
        # the warm-up window.  At step=0 the agent sees only net-return
        # scaled reward; at step=dsr_warmup_steps it sees only DSR reward.
        # This avoids the abrupt reward discontinuity of a hard switch.
        if self.dsr_warmup_steps > 0 and self._step < self.dsr_warmup_steps:
            alpha  = self._step / self.dsr_warmup_steps          # 0 → 1
            reward = ((1.0 - alpha) * self.reward_scale_net * net
                      + alpha       * self.reward_scale_dsr * dsr_val)
        else:
            reward = self.reward_scale_dsr * dsr_val

        self._step += 1; self._w_prev = w_new; self._t += 1

        ew   = 1.0 / self.N
        info = {
            "net_ret": net, "gross_ret": gross, "cost": cost, "turnover": turnover,
            "dsr": dsr_val, "w": w_new.astype(np.float32),
            "l1_to_ew": float(np.sum(np.abs(w_new - ew))),
            "went": self._entropy(w_new), "wmax": float(np.max(w_new)),
            "tradable_n": int(mask.sum()),
        }
        return self._get_obs(), float(reward), False, False, info


---
## 10 · Metrics & Plotting


In [ ]:
def annualized_sharpe(returns: np.ndarray, eps: float = 1e-12, periods_per_year: int = 252) -> float:
    r = np.asarray(returns, dtype=np.float64)
    if r.size < 10: return float("nan")
    mu = float(np.mean(r)); sd = float(np.std(r))
    if sd < eps: return float("nan")
    return float((mu / sd) * math.sqrt(periods_per_year))

def annualized_return(returns: np.ndarray, periods_per_year: int = 252) -> float:
    r = np.asarray(returns, dtype=np.float64)
    if r.size == 0: return float("nan")
    wealth = np.prod(1.0 + r); years = len(r) / periods_per_year
    if years <= 0 or wealth <= 0: return float("nan")
    return float(wealth ** (1.0 / years) - 1.0)

def max_drawdown(returns: np.ndarray) -> float:
    wealth = _cum_wealth(returns)
    peak   = np.maximum.accumulate(wealth)
    return float(np.min(wealth / peak - 1.0))

def _cum_wealth(rets: np.ndarray) -> np.ndarray:
    return np.cumprod(1.0 + np.asarray(rets, dtype=np.float64))

def rollout(model: PPO, env: gym.Env, deterministic: bool = True,
            dates_index: Optional[pd.DatetimeIndex] = None) -> Dict[str, Any]:
    obs, _ = env.reset(); done = False
    nets, grosses, costs, turns, ws, l1s, ents, wmaxs, tradables = ([] for _ in range(9))
    step_ts: List[int] = []
    while not done:
        step_t = int(env._t)  # absolute index before step — date for this allocation
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, _, done, trunc, info = env.step(action)
        nets.append(info["net_ret"]); grosses.append(info["gross_ret"])
        costs.append(info["cost"]);   turns.append(info["turnover"])
        ws.append(info["w"]);         l1s.append(info["l1_to_ew"])
        step_ts.append(step_t)
        ents.append(info["went"]);    wmaxs.append(info["wmax"])
        tradables.append(info["tradable_n"])
        if trunc: break
    nets = np.asarray(nets, dtype=np.float32); ws = np.asarray(ws, dtype=np.float32)
    dates_out = (dates_index[step_ts] if dates_index is not None and len(step_ts) > 0
                else None)
    return {
        "net": nets, "gross": np.asarray(grosses, dtype=np.float32), "w": ws,
        "sharpe": annualized_sharpe(nets), "ann_return": annualized_return(nets),
        "max_drawdown": max_drawdown(nets),
        "mu": float(nets.mean()), "sd": float(nets.std()),
        "cost_mu": float(np.mean(costs)), "turnover_mu": float(np.mean(turns)),
        "l1_mu": float(np.mean(l1s)), "ent_mu": float(np.mean(ents)),
        "wmax_mu": float(np.mean(wmaxs)), "tradable_n_mu": float(np.mean(tradables)),
        "n": int(len(nets)),
        "dates": dates_out,
    }

def rollout_equal_weight(env: "OneDayHoldPortfolioEnv") -> Dict[str, Any]:
    obs, _ = env.reset(); done = False; nets, grosses = [], []
    action = np.zeros(env.N, dtype=np.float32)
    while not done:
        obs, _, done, trunc, info = env.step(action)
        nets.append(info["net_ret"]); grosses.append(info["gross_ret"])
        if trunc: break
    nets = np.asarray(nets, dtype=np.float32)
    return {
        "net": nets, "gross": np.asarray(grosses, dtype=np.float32),
        "sharpe": annualized_sharpe(nets), "ann_return": annualized_return(nets),
        "max_drawdown": max_drawdown(nets), "mu": float(nets.mean()),
        "sd": float(nets.std()), "n": int(len(nets)),
    }

def save_plots(out_dir: str, prefix: str, agent: Dict[str, Any], ew: Dict[str, Any], asset_names: List[str]):
    os.makedirs(out_dir, exist_ok=True)

    plt.figure()
    plt.plot(_cum_wealth(agent["net"]), label="agent")
    plt.plot(_cum_wealth(ew["net"]),    label="equal_weight")
    plt.title(f"{prefix} cumulative wealth"); plt.xlabel("step"); plt.ylabel("wealth")
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_cum.png")); plt.close()

    if "w" in agent and agent["w"].ndim == 2 and agent["w"].shape[0] > 5:
        plt.figure()
        plt.stackplot(np.arange(agent["w"].shape[0]), agent["w"].T, labels=asset_names)
        plt.title(f"{prefix} weights"); plt.xlabel("step"); plt.ylabel("weight")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"{prefix}_weights.png")); plt.close()

    plt.figure()
    plt.plot(agent["net"], label="agent"); plt.plot(ew["net"], label="equal_weight")
    plt.title(f"{prefix} daily net returns"); plt.xlabel("step"); plt.ylabel("return")
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_daily_returns.png")); plt.close()

def save_metrics_json(path: str, metrics: Dict[str, Any]):
    out = {}
    for k, v in metrics.items():
        if isinstance(v, (np.ndarray, pd.DatetimeIndex)): continue
        out[k] = float(v) if isinstance(v, (np.floating, np.integer)) else v
    with open(path, "w") as f: json.dump(out, f, indent=2)


---
## 11 · Training Callbacks

- **`EarlyStopSharpeCallback`** — evaluates on the early-stop validation split every `eval_freq` steps and restores the best checkpoint when patience is exhausted.
- **`DiagnosticsCallback`** — periodically logs train / ES-val / test Sharpe ratios to console and TensorBoard.


In [ ]:
class EarlyStopSharpeCallback(BaseCallback):
    def __init__(
        self, eval_env_fn, eval_freq: int, patience_evals: int,
        min_evals: int = 5, min_delta: float = 0.0,
        deterministic: bool = True, verbose: int = 0,
    ):
        super().__init__(verbose=verbose)
        self.eval_env_fn    = eval_env_fn
        self.eval_freq      = int(eval_freq)
        self.patience_evals = int(patience_evals)
        self.min_evals      = int(min_evals)
        self.min_delta      = float(min_delta)
        self.deterministic  = deterministic

        self.best_sharpe         = -np.inf
        self.best_num_timesteps  = 0
        self.no_improve_count    = 0
        self.eval_count          = 0
        self.best_state_dict     = None

    def _on_step(self) -> bool:
        if self.num_timesteps % self.eval_freq != 0:
            return True
        self.eval_count += 1
        sh = float(rollout(self.model, self.eval_env_fn(), deterministic=self.deterministic)["sharpe"])

        self.logger.record("early_stop/es_sharpe", sh)
        self.logger.record("early_stop/best_es_sharpe",
                           self.best_sharpe if np.isfinite(self.best_sharpe) else np.nan)
        self.logger.record("early_stop/no_improve_count", self.no_improve_count)
        self.logger.dump(self.num_timesteps)

        if np.isfinite(sh) and sh > self.best_sharpe + self.min_delta:
            self.best_sharpe        = sh
            self.best_num_timesteps = self.num_timesteps
            self.no_improve_count   = 0
            self.best_state_dict    = {k: v.detach().cpu().clone()
                                       for k, v in self.model.policy.state_dict().items()}
            if self.verbose: print(f"[ES] step={self.num_timesteps} new best sharpe={sh:.4f}")
        else:
            self.no_improve_count += 1
            if self.verbose:
                print(f"[ES] step={self.num_timesteps} sharpe={sh:.4f} "
                      f"best={self.best_sharpe:.4f} no_improve={self.no_improve_count}/{self.patience_evals}")

        if self.eval_count >= self.min_evals and self.no_improve_count >= self.patience_evals:
            if self.verbose:
                print(f"[ES STOP] step={self.num_timesteps} "
                      f"best_sharpe={self.best_sharpe:.4f} at step={self.best_num_timesteps}")
            return False
        return True

    def restore_best_model(self):
        if self.best_state_dict is not None:
            self.model.policy.load_state_dict(self.best_state_dict)


class DiagnosticsCallback(BaseCallback):
    def __init__(
        self, eval_every_steps: int, train_env_fn, es_val_env_fn, test_env_fn,
        name: str, deterministic: bool = True, verbose: int = 0,
    ):
        super().__init__(verbose=verbose)
        self.eval_every_steps = int(eval_every_steps)
        self.train_env_fn     = train_env_fn
        self.es_val_env_fn    = es_val_env_fn
        self.test_env_fn      = test_env_fn
        self.name             = name
        self.deterministic    = deterministic

    def _on_step(self) -> bool:
        if self.num_timesteps % self.eval_every_steps != 0:
            return True
        tr = rollout(self.model, self.train_env_fn(),  deterministic=self.deterministic)
        es = rollout(self.model, self.es_val_env_fn(), deterministic=self.deterministic)
        te = rollout(self.model, self.test_env_fn(),   deterministic=self.deterministic)
        print(f"[{self.name}] step={self.num_timesteps} "
              f"TR sh={tr['sharpe']:.3f} | ES sh={es['sharpe']:.3f} | TEST sh={te['sharpe']:.3f}")
        self.logger.record("diag/train_sharpe",   tr["sharpe"])
        self.logger.record("diag/es_val_sharpe",  es["sharpe"])
        self.logger.record("diag/test_sharpe",    te["sharpe"])
        self.logger.dump(self.num_timesteps)
        return True


---
## 12 · Low-Level Trainer

`_build_and_train_ppo` is the single function that constructs the PPO model, runs `model.learn()`, and restores the best early-stopping checkpoint.  
It is used by both the Optuna objective and the final run — keeping the training logic in one place.


In [ ]:
def make_vec_env(env_ctor):
    return DummyVecEnv([env_ctor])


def _build_and_train_ppo(
    vec_train,
    ppo_params:       Dict[str, Any],
    total_timesteps:  int,
    es_callback:      EarlyStopSharpeCallback,
    diag_callback:    DiagnosticsCallback,
    seed:             int,
    tb_path:          str,
) -> PPO:
    """Construct, train, and return a PPO model with best ES weights restored."""
    np.random.seed(seed); random.seed(seed); th.manual_seed(seed)
    #os.makedirs(tb_path, exist_ok=True)
    #sb3_logger = sb3_configure_logger(tb_path, ["csv", "tensorboard"])

    model = PPO(
        "MlpPolicy", vec_train, seed=seed, verbose=0,
        #tensorboard_log=tb_path,
        policy_kwargs=ppo_params["policy_kwargs"],
        gamma=ppo_params["gamma"], gae_lambda=ppo_params["gae_lambda"],
        learning_rate=ppo_params["learning_rate"], clip_range=ppo_params["clip_range"],
        n_epochs=ppo_params["n_epochs"], n_steps=ppo_params["n_steps"],
        batch_size=ppo_params["batch_size"], ent_coef=ppo_params["ent_coef"],
        vf_coef=ppo_params["vf_coef"], max_grad_norm=ppo_params["max_grad_norm"],
        target_kl=ppo_params["target_kl"],
    )
    #model.set_logger(sb3_logger)
    model.learn(
        total_timesteps=total_timesteps,
        callback=CallbackList([es_callback, diag_callback]),
        progress_bar=False,
    )
    es_callback.restore_best_model()
    return model


---
## 13 · Search-Space Size Estimator

A quick sanity-check: how large is the discrete part of the search space, and how many trials should we run?


In [ ]:
def estimate_search_space(study: optuna.Study) -> int:
    """
    Returns the number of discrete combinations from the study's search space.
    Continuous params (float/int) are noted but excluded from the count.
    """
    if not study.trials:
        raise ValueError("Run at least 1 trial first so distributions are populated.")
    distributions = study.trials[0].distributions
    discrete_combinations = 1
    continuous_params: List[str] = []
    for name, dist in distributions.items():
        if isinstance(dist, optuna.distributions.CategoricalDistribution):
            discrete_combinations *= len(dist.choices)
        elif isinstance(dist, optuna.distributions.IntDistribution):
            discrete_combinations *= (dist.high - dist.low) // dist.step + 1
        else:
            # FloatDistribution — infinite, just log it
            continuous_params.append(name)
    print(f"Discrete combinations : {discrete_combinations}")
    print(f"Continuous params     : {continuous_params} (excluded from count)")
    print(f"Suggested N_TRIALS    : {math.ceil(math.sqrt(discrete_combinations))}")
    return discrete_combinations


---
## 14 · Optuna Objective & `run_optuna_search`

The objective function wires together feature selection, HPO, walk-forward splits, and the low-level trainer.  
The reported score is the **mean Sharpe ratio across all (split × seed) combinations** — rewarding robustness over any single fold.

> **Note on `cost_mult`:** removed. Spreads are empirical bid-ask estimates — treating cost scaling as a tunable hyperparameter would be economically unsound.


In [ ]:
def objective_factory(
    df:           pd.DataFrame,
    assets:       List[str],
    features:     Dict[str, List[str]],
    spreads:      np.ndarray,
    base_log_dir: str,
    seeds:        List[int],
):
    dates_index = pd.DatetimeIndex(df.index)
    splits      = make_walk_forward_splits(dates_index)

    def objective(trial: optuna.Trial) -> float:
        trial_dir = os.path.join(base_log_dir, f"trial_{trial.number:05d}")
        #os.makedirs(trial_dir, exist_ok=True)
        logger = setup_logger(trial_dir, name_prefix="ppo_tune")

        logger.info("=" * 80)
        logger.info(f"TRIAL {trial.number} START | seeds={seeds} splits={len(splits)}")

        # ── Feature selection ────────────────────────────────────────────────
        picks = pick_features_from_buckets(trial, features)
        per_asset_feats = get_per_asset_feature_list(picks)
        trial.set_user_attr("feature_set",     picks)
        trial.set_user_attr("per_asset_feats", per_asset_feats)

        # ── PPO hyperparameters ──────────────────────────────────────────────
        gamma           = trial.suggest_float("gamma",         0.90, 0.999)
        gae_lambda      = trial.suggest_float("gae_lambda",    0.80, 0.98)
        learning_rate   = trial.suggest_float("learning_rate", 1e-5, 2e-3, log=True)
        clip_range      = trial.suggest_float("clip_range",    0.05, 0.20)
        n_epochs        = trial.suggest_int(  "n_epochs",      3, 10)

        n_steps     = trial.suggest_categorical("n_steps_v2",   [252, 504, 756])
        BATCH_SPACE = [63, 84, 108, 126, 168, 189, 252, 378, 504, 756]
        batch_size  = trial.suggest_categorical("batch_size_v2", BATCH_SPACE)

        total_timesteps = 1_000_000
        trial.set_user_attr("total_timesteps", total_timesteps)

        valid_batch_sizes = {
            252: {63, 84, 126, 252},
            504: {84, 126, 168, 252, 504},
            756: {63, 84, 108, 126, 189, 252, 378, 756},
        }
        if batch_size not in valid_batch_sizes[n_steps] or batch_size > n_steps:
            raise optuna.exceptions.TrialPruned()

        target_kl     = trial.suggest_float("target_kl",     0.003, 0.03)
        ent_coef      = trial.suggest_float("ent_coef",      1e-5, 5e-2, log=True)
        vf_coef       = trial.suggest_float("vf_coef",       0.1, 1.0)
        max_grad_norm = trial.suggest_float("max_grad_norm", 0.3, 2.0)

        dsr_eta           = trial.suggest_categorical("dsr_eta",          [round(1/252, 6), 0.005, 0.01, 0.02])
        dsr_warmup_steps  = trial.suggest_categorical("dsr_warmup_steps", [0, 126, 252, 504])
        reward_scale_net  = trial.suggest_categorical("reward_scale_net", [50.0, 100.0, 200.0])
        reward_scale_dsr  = trial.suggest_categorical("reward_scale_dsr", [5.0, 10.0, 20.0])
        dsr_clip          = trial.suggest_categorical("dsr_clip",         [5.0, 10.0, 20.0])

        lookback     = trial.suggest_categorical("lookback",     [5, 22, 60])
        encoder_width = trial.suggest_categorical("encoder_width", [32, 64, 128, 256])
        log_std_init = trial.suggest_float("log_std_init", -1.5, 0.0)

        trial.set_user_attr("lookback", lookback)

        has_market_flag = (picks.get("Market") is not None)
        n_feats = len(get_per_asset_feature_list(picks))
        policy_kwargs = dict(
            features_extractor_class=PerAssetFeaturesExtractor,
            features_extractor_kwargs=dict(
                n_assets=len(assets),
                lookback=lookback,
                n_features=n_feats,
                has_market=has_market_flag,
                encoder_width=encoder_width,
            ),
            net_arch=[encoder_width, encoder_width],
            activation_fn=nn.Tanh, log_std_init=log_std_init)
        ppo_params = dict(
            gamma=gamma, gae_lambda=gae_lambda, learning_rate=learning_rate,
            clip_range=clip_range, n_epochs=n_epochs, n_steps=n_steps, batch_size=batch_size,
            target_kl=target_kl, ent_coef=ent_coef, vf_coef=vf_coef, max_grad_norm=max_grad_norm,
            policy_kwargs=policy_kwargs, dsr_warmup_steps=dsr_warmup_steps,
            reward_scale_net=reward_scale_net, reward_scale_dsr=reward_scale_dsr, dsr_clip=dsr_clip,
        )

        logger.info(f"Feature picks: {picks} | Lookback: {lookback}")
        logger.info(f"PPO params: {ppo_params}")

        closes_raw, X_raw, feat_names, market_raw = build_raw_arrays(
            df, assets, picks, features, logger)
        avail = compute_availability_mask(closes_raw, X_raw, lookback=lookback)

        trial.set_user_attr("feat_names",  feat_names)
        trial.set_user_attr("has_market",  int(market_raw is not None))

        all_scores: List[float] = []

        for split_i, (train_idx, val_idx) in enumerate(splits):
            logger.info("-" * 80)
            logger.info(f"[SPLIT {split_i}] "
                        f"train {df.index[train_idx[0]]}..{df.index[train_idx[-1]]} "
                        f"val {df.index[val_idx[0]]}..{df.index[val_idx[-1]]}")

            X_train_raw      = X_raw[train_idx]
            market_train_raw = market_raw[train_idx] if market_raw is not None else None
            fitted, market_ft = fit_transforms_on_train(X_train_raw, feat_names, market_train_raw)
            X_trans, market_trans = apply_transforms(X_raw, fitted, market_raw, market_ft)

            tr_ratio = float(avail[train_idx].mean()); va_ratio = float(avail[val_idx].mean())
            logger.info(f"[AVAIL] train={tr_ratio:.3f} val={va_ratio:.3f}")

            if float(avail[val_idx].sum(axis=1).mean()) < 2.0:
                logger.info("[PRUNE] Too few tradable assets on validation.")
                raise optuna.exceptions.TrialPruned()

            seed_scores: List[float] = []
            for seed in seeds:
                train_core_idx, es_val_idx = split_train_for_early_stop(
                    train_idx, dates_index, es_years=1, min_train_days=252)

                tr_s = max(int(train_core_idx[0]), lookback - 1)
                tr_e = min(int(train_core_idx[-1]), closes_raw.shape[0] - 2)
                es_s = max(int(es_val_idx[0]),   lookback - 1)
                es_e = min(int(es_val_idx[-1]),  closes_raw.shape[0] - 2)
                va_s = max(int(val_idx[0]),       lookback - 1)
                va_e = min(int(val_idx[-1]),      closes_raw.shape[0] - 2)

                def _env(s, e):
                    return OneDayHoldPortfolioEnv(
                        closes_raw=closes_raw, feats=X_trans, market=market_trans,
                        avail_mask=avail, spreads=spreads,
                        dsr_eta=dsr_eta, dsr_clip=dsr_clip,
                        dsr_warmup_steps=dsr_warmup_steps,
                        reward_scale_net=reward_scale_net, reward_scale_dsr=reward_scale_dsr,
                        start_idx=s, end_idx=e, lookback=lookback,
                    )

                es_cb = EarlyStopSharpeCallback(
                    eval_env_fn=lambda s=es_s, e=es_e: _env(s, e),
                    eval_freq=50_000, patience_evals=4, min_evals=5, verbose=1)
                diag_cb = DiagnosticsCallback(
                    eval_every_steps=50_000,
                    train_env_fn   =lambda s=tr_s, e=tr_e: _env(s, e),
                    es_val_env_fn  =lambda s=es_s, e=es_e: _env(s, e),
                    test_env_fn    =lambda s=va_s, e=va_e: _env(s, e),
                    name=f"split{split_i}_seed{seed}")

                tb_path = os.path.join(trial_dir, f"tb_split{split_i}_seed{seed}")
                model = _build_and_train_ppo(
                    vec_train=make_vec_env(lambda s=tr_s, e=tr_e: _env(s, e)),
                    ppo_params=ppo_params, total_timesteps=total_timesteps,
                    es_callback=es_cb, diag_callback=diag_cb, seed=seed, tb_path=tb_path,
                )

                sh = float(rollout(model, _env(va_s, va_e))["sharpe"])
                logger.info(f"[EVAL] split={split_i} seed={seed} val_sharpe={sh:.4f}")
                seed_scores.append(sh); all_scores.append(sh)

            logger.info(f"[SPLIT {split_i}] per_seed={seed_scores} mean={np.mean(seed_scores):.4f}")
            trial.report(float(np.mean(seed_scores)), step=split_i)
            if trial.should_prune():
                logger.info("[PRUNE] Optuna pruned this trial.")
                raise optuna.TrialPruned()

        score = float(np.mean(all_scores))
        trial.set_user_attr("all_scores", [float(x) for x in all_scores])
        logger.info("=" * 80)
        logger.info(f"TRIAL {trial.number} DONE  score={score:.4f}  all_scores={all_scores}")
        return score

    return objective


def run_optuna_search(
    df:        pd.DataFrame,
    assets:    List[str],
    features:  Dict[str, List[str]],
    spreads:   np.ndarray,
    out_dir:   str,
    n_trials:  int = 50,
    seeds:     Optional[List[int]] = None,
) -> Tuple[optuna.Study, str]:
    """
    Run the Optuna HPO search and return the completed study + output directory.

    After this completes, call `estimate_search_space(study)` to gauge coverage,
    then proceed to `run_final_train_test` with the best parameters.
    """
    os.makedirs(out_dir, exist_ok=True)
    seeds = seeds or [11, 22, 33]

    storage_path = os.path.join(out_dir, "ppo_stationary_all.db")
    storage      = f"sqlite:///{storage_path}"

    study = optuna.create_study(
        study_name="ppo_stationary_fixed_setup",
        direction="maximize",
        storage=storage,
        load_if_exists=True,
        sampler=TPESampler(seed=None, multivariate=True, group=True, constant_liar=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1),
    )

    objective = objective_factory(
        df=df, assets=assets, features=features,
        spreads=spreads, base_log_dir=out_dir, seeds=seeds,
    )
    study.optimize(objective, n_trials=n_trials, gc_after_trial=True, show_progress_bar=True)

    best = study.best_trial
    summary = {
        "best_value":      best.value,
        "best_params":     best.params,
        "best_user_attrs": best.user_attrs,
        "n_trials":        len(study.trials),
    }
    with open(os.path.join(out_dir, "best_trial.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print("\n=== BEST TRIAL ===")
    print("Value:", best.value)
    print("Params:", best.params)
    print("Feature set:", best.user_attrs.get("feature_set"))
    print("DB:", storage_path)
    return study, out_dir


---
## 15 · Run Tuning

Configure the dataset path, assets, feature buckets, and spread file below, then execute.

After the study completes, `estimate_search_space(study)` reports the discrete search-space size and the suggested number of trials (≈ √(discrete combinations)).


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
resultsfolder = "v913"

df = pd.read_parquet(str(PROJECT_ROOT / "03_portfolio/dataset.parquet"), engine="pyarrow")

assets = ['XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']

features = {
    "Price":      ["Close"],
    "OHLC":       ["Open", "High", "Low"],
    "Trend":      ["SMA5", "SMA22", "SMA63", "SMA126", "SMA252", "MACD", "ADX"],
    "Momentum":   ["RSI", "MACD", "CCI"],
    "Volume":     ["Volume", "OBV", "ADI", "VolumeVariation"],
    "Volatility": ["BBUpper", "BBLower", "5dayVolatility", "22dayVolatility",
                   "63dayVolatility", "RatioVolatility"],
    "Market":     ["VIXIndexClose"],
}

with open(str(PROJECT_ROOT / "01_data/aux/bid-ask_spread.json")) as f:
    spread_dict = json.load(f)
spreads = np.array([spread_dict[a] for a in assets], dtype=np.float32)

# ── Run search ─────────────────────────────────────────────────────────────
study, log_dir = run_optuna_search(
    df=df,
    assets=assets,
    features=features,
    spreads=spreads,
    out_dir=resultsfolder,
    n_trials=0,
    seeds=[random.randint(0, 2**32 - 1) for _ in range(3)],
)

In [ ]:
# ── Search-space diagnostics ───────────────────────────────────────────────
estimate_search_space(study)

print(f"You have ran {len(study.trials)} trials.")


---
## 16 · Load Best Trial

Reload the completed Optuna study and extract the best configuration. The cell below also pretty-prints the chosen feature set and PPO parameters for review before committing to the final run.


In [ ]:
from pprint import pprint

storage_path = f"{resultsfolder}/ppo_stationary_all.db"
study = optuna.load_study(
    study_name="ppo_stationary_fixed_setup",
    storage=f"sqlite:///{storage_path}",
)

completed_trials = [
    t for t in study.trials
    if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None
]
sorted_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)
best       = sorted_trials[0]
params     = best.params
user_attrs = best.user_attrs

feature_picks    = user_attrs["feature_set"]
lookback         = user_attrs["lookback"]
total_timesteps  = user_attrs.get("total_timesteps", 1_000_000)
dsr_eta          = params["dsr_eta"]

encoder_width = params["encoder_width"]
log_std_init = params["log_std_init"]

ppo_params = {
    "gamma":          params["gamma"],
    "gae_lambda":     params["gae_lambda"],
    "learning_rate":  params["learning_rate"],
    "clip_range":     params["clip_range"],
    "n_epochs":       params["n_epochs"],
    "n_steps":        params["n_steps_v2"],
    "batch_size":     params["batch_size_v2"],
    "target_kl":      params["target_kl"],
    "ent_coef":       params["ent_coef"],
    "vf_coef":        params["vf_coef"],
    "max_grad_norm":  params["max_grad_norm"],
    "_encoder_width":  encoder_width,  # stored for reference
    # policy_kwargs is rebuilt in run_final_train_test from feature_picks
    "log_std_init":     log_std_init,   # stored here so run_final can pop it
    "dsr_warmup_steps": params["dsr_warmup_steps"],
    "reward_scale_net": params["reward_scale_net"],
    "reward_scale_dsr": params["reward_scale_dsr"],
    "dsr_clip":         params["dsr_clip"],
}

print(f"Best trial #{best.number}  value={best.value:.4f}  "
      f"({len(study.trials)} total trials)")
print("\nBEST_FEATURE_PICKS = ", end=""); pprint(feature_picks)
print("\nBEST_PPO_PARAMS = ",    end=""); pprint(ppo_params)
print(f"\nBEST_DSR_ETA   = {dsr_eta}")
print(f"BEST_LOOKBACK   = {lookback}")
print(f"BEST_TIMESTEPS  = {total_timesteps}")


---
## 17 · Final Run Functions

`run_final_train_test` orchestrates the complete train → (optional) mid-test refit → test pipeline:

1. **Train** on 2015-07-01 – 2023-06-30 with early stopping on the last year.  
2. **Test (first half)** 2023-07-01 – 2024-06-30 using the trained model.  
3. **Refit** on all data up to 2024-06-30 (optional, enabled by default).  
4. **Test (second half)** 2024-07-01 – 2025-06-30 using the refitted model.  

The function returns all weights (train + both test halves) as arrays, which can be fed directly to `PortfolioMetrics`.


In [ ]:
def train_and_eval_final(
    closes_raw:        np.ndarray,
    X_transformed:     np.ndarray,
    market_transformed: Optional[np.ndarray],
    avail_mask:        np.ndarray,
    spreads:           np.ndarray,
    asset_names:       List[str],
    train_idx:         np.ndarray,
    test_idx:          np.ndarray,
    dates_index:       pd.DatetimeIndex,
    seed:              int,
    ppo_params:        Dict[str, Any],
    dsr_eta:           float,
    total_timesteps:   int,
    out_dir:           str,
    logger:            logging.Logger,
    lookback:          int,
    refit_train_idx:   Optional[np.ndarray] = None,  # for mid-test refit
    label:             str = "final",
) -> Dict[str, Any]:
    """
    Train PPO on `train_idx`, evaluate on `test_idx`.
    If `refit_train_idx` is supplied this is the refit pass.
    """
    train_core_idx, es_val_idx = split_train_for_early_stop(
        refit_train_idx if refit_train_idx is not None else train_idx,
        dates_index, es_years=1, min_train_days=252,
    )

    tr_s = max(int(train_core_idx[0]),  lookback - 1)
    tr_e = min(int(train_core_idx[-1]), closes_raw.shape[0] - 2)
    es_s = max(int(es_val_idx[0]),      lookback - 1)
    es_e = min(int(es_val_idx[-1]),     closes_raw.shape[0] - 2)
    te_s = max(int(test_idx[0]),        lookback - 1)
    te_e = min(int(test_idx[-1]),       closes_raw.shape[0] - 2)

    def _env(s, e):
        return OneDayHoldPortfolioEnv(
            closes_raw=closes_raw, feats=X_transformed, market=market_transformed,
            avail_mask=avail_mask, spreads=spreads,
            dsr_eta=dsr_eta, dsr_clip=ppo_params["dsr_clip"],
            dsr_warmup_steps=ppo_params["dsr_warmup_steps"],
            reward_scale_net=ppo_params["reward_scale_net"],
            reward_scale_dsr=ppo_params["reward_scale_dsr"],
            start_idx=s, end_idx=e, lookback=lookback,
        )

    es_cb = EarlyStopSharpeCallback(
        eval_env_fn=lambda: _env(es_s, es_e),
        eval_freq=50_000, patience_evals=4, min_evals=5, verbose=1)
    diag_cb = DiagnosticsCallback(
        eval_every_steps=50_000,
        train_env_fn  =lambda: _env(tr_s, tr_e),
        es_val_env_fn =lambda: _env(es_s, es_e),
        test_env_fn   =lambda: _env(te_s, te_e),
        name=label)

    tb_path = os.path.join(out_dir, f"tb_{label}")
    logger.info(f"[{label.upper()}] "
                f"train_core={dates_index[train_core_idx[0]]}"
                f"..{dates_index[train_core_idx[-1]]} "
                f"es_val={dates_index[es_val_idx[0]]}"
                f"..{dates_index[es_val_idx[-1]]} "
                f"test={dates_index[test_idx[0]]}"
                f"..{dates_index[test_idx[-1]]}")

    model = _build_and_train_ppo(
        vec_train=make_vec_env(lambda: _env(tr_s, tr_e)),
        ppo_params=ppo_params, total_timesteps=total_timesteps,
        es_callback=es_cb, diag_callback=diag_cb, seed=seed, tb_path=tb_path,
    )

    train_metrics = rollout(model, _env(tr_s, tr_e), dates_index=dates_index)
    es_metrics    = rollout(model, _env(es_s, es_e), dates_index=dates_index)
    test_metrics  = rollout(model, _env(te_s, te_e), dates_index=dates_index)
    ew_metrics    = rollout_equal_weight(_env(te_s, te_e))

    logger.info(f"[{label.upper()} EVAL] "
                f"test_sharpe={test_metrics['sharpe']:.4f} "
                f"ann_ret={test_metrics['ann_return']:.4%} "
                f"mdd={test_metrics['max_drawdown']:.4%} "
                f"best_es_sharpe={es_cb.best_sharpe:.4f}")

    model_path = os.path.join(out_dir, f"ppo_{label}_model.zip")
    model.save(model_path)

    save_plots(os.path.join(out_dir, f"plots_{label}"), label,
               test_metrics, ew_metrics, asset_names)
    save_metrics_json(os.path.join(out_dir, f"{label}_test_metrics.json"),  test_metrics)
    save_metrics_json(os.path.join(out_dir, f"{label}_train_metrics.json"), train_metrics)
    save_metrics_json(os.path.join(out_dir, f"{label}_ew_metrics.json"),    ew_metrics)

    return {
        "model": model, "model_path": model_path,
        "best_es_sharpe": float(es_cb.best_sharpe),
        "train": train_metrics, "es": es_metrics,
        "test": test_metrics, "ew_test": ew_metrics,
    }


def run_final_train_test(
    df:              pd.DataFrame,
    assets:          List[str],
    spreads:         np.ndarray,
    feature_picks:   Dict[str, Any],
    ppo_params:      Dict[str, Any],
    dsr_eta:         float,
    lookback:        int,
    out_dir:         str,
    total_timesteps: int = 1_000_000,
    seed:            int = SEED_PORTFOLIO_DRL_PPO_V9_1_3,
    refit_mid_date:  str = "2024-07-01",
    do_refit:        bool = True,
) -> Dict[str, Any]:
    """
    Full pipeline:
      1. Train on 2015-07-01 – 2023-06-30
      2. Test first half  2023-07-01 – (refit_mid_date - 1 day)
      3. [Optional] Refit on all data up to refit_mid_date - 1 day
      4. Test second half refit_mid_date – 2025-06-30

    Returns a summary dict with all weights stacked as
        summary["all_weights"]  shape [T_train + T_test, N]  (numpy)
        summary["weights_df"]   DatetimeIndex DataFrame [T_train + T_test, N]
        summary["train_weights_df"]  train slice of weights_df
        summary["test_weights_df"]   test slice of weights_df
    ready for PortfolioMetrics.
    """
    os.makedirs(out_dir, exist_ok=True)
    logger = setup_logger(out_dir, name_prefix="ppo_final")
    logger.info("=" * 100)
    logger.info("FINAL TRAIN/TEST RUN START")
    logger.info(f"feature_picks={feature_picks}  lookback={lookback}  "
                f"dsr_eta={dsr_eta}  seed={seed}  refit={do_refit}@{refit_mid_date}")

    dates_index = pd.DatetimeIndex(df.index)
    train_idx, test_idx = make_final_train_test_split(dates_index)

    features_dummy: Dict[str, List[str]] = {}   # not needed — picks already resolved
    closes_raw, X_raw, feat_names, market_raw = build_raw_arrays(
        df, assets, feature_picks, features_dummy, logger)
    avail = compute_availability_mask(closes_raw, X_raw, lookback=lookback)

    # Fit transforms on the initial training window
    fitted, market_ft = fit_transforms_on_train(
        X_raw[train_idx],
        feat_names,
        market_raw[train_idx] if market_raw is not None else None,
    )
    X_trans, market_trans = apply_transforms(X_raw, fitted, market_raw, market_ft)

    logger.info(f"TRAIN = {dates_index[train_idx[0]]} .. {dates_index[train_idx[-1]]}")
    logger.info(f"TEST  = {dates_index[test_idx[0]]}  .. {dates_index[test_idx[-1]]}")

    if do_refit:
        test_first_idx, test_second_idx = make_midtest_refit_split(
            test_idx, dates_index, mid_date=refit_mid_date)
    else:
        test_first_idx  = test_idx
        test_second_idx = None

    # ── Rebuild policy_kwargs with PerAssetFeaturesExtractor ──────────────────
    per_asset_feats = get_per_asset_feature_list(feature_picks)
    has_market_flag = (feature_picks.get('Market') is not None)
    encoder_width_  = ppo_params.pop('_encoder_width', ppo_params.get('net_arch', [64])[0])
    # Extract log_std_init from tuned params (stored directly in ppo_params
    # by the Optuna objective as a float from suggest_float).
    log_std_init_   = ppo_params.pop('log_std_init', -1.0)
    ppo_params['policy_kwargs'] = dict(
        features_extractor_class=PerAssetFeaturesExtractor,
        features_extractor_kwargs=dict(
            n_assets=len(assets),
            lookback=lookback,
            n_features=len(per_asset_feats),
            has_market=has_market_flag,
            encoder_width=encoder_width_,
        ),
        net_arch=[encoder_width_, encoder_width_],
        activation_fn=nn.Tanh,
        log_std_init=log_std_init_,
    )

    # ── Phase 1: train & evaluate on test_first ──────────────────────────────
    result1 = train_and_eval_final(
        closes_raw=closes_raw, X_transformed=X_trans,
        market_transformed=market_trans, avail_mask=avail,
        spreads=spreads, asset_names=assets,
        train_idx=train_idx, test_idx=test_first_idx,
        dates_index=dates_index, seed=seed,
        ppo_params=ppo_params, dsr_eta=dsr_eta,
        total_timesteps=total_timesteps, out_dir=out_dir,
        logger=logger, lookback=lookback, label="phase1",
    )

    all_test_nets  = [result1["test"]["net"]]
    all_test_w     = [result1["test"]["w"]]
    all_ew_nets    = [result1["ew_test"]["net"]]

    # ── Phase 2: refit up to mid_date and evaluate on test_second ────────────
    if do_refit and test_second_idx is not None:
        # Refit train window = original train + test_first
        refit_train_idx = np.concatenate([train_idx, test_first_idx])

        # Re-fit transforms on the extended training window
        fitted2, market_ft2 = fit_transforms_on_train(
            X_raw[refit_train_idx], feat_names,
            market_raw[refit_train_idx] if market_raw is not None else None,
        )
        X_trans2, market_trans2 = apply_transforms(X_raw, fitted2, market_raw, market_ft2)

        logger.info(f"REFIT train window up to {dates_index[refit_train_idx[-1]]}")

        result2 = train_and_eval_final(
            closes_raw=closes_raw, X_transformed=X_trans2,
            market_transformed=market_trans2, avail_mask=avail,
            spreads=spreads, asset_names=assets,
            train_idx=train_idx, test_idx=test_second_idx,
            dates_index=dates_index, seed=seed,
            ppo_params=ppo_params, dsr_eta=dsr_eta,
            total_timesteps=total_timesteps, out_dir=out_dir,
            logger=logger, lookback=lookback,
            refit_train_idx=refit_train_idx, label="phase2",
        )
        all_test_nets.append(result2["test"]["net"])
        all_test_w.append(result2["test"]["w"])
        all_ew_nets.append(result2["ew_test"]["net"])
    else:
        result2 = None

    # ── Combine nets and weights arrays ──────────────────────────────────────
    combined_test_nets = np.concatenate(all_test_nets)
    combined_test_w    = np.vstack(all_test_w)
    combined_ew_nets   = np.concatenate(all_ew_nets)

    # ── Build dated weight DataFrames from rollout dates ──────────────────────
    # dates come directly from rollout() — no post-hoc index arithmetic needed
    train_dates = result1["train"]["dates"]
    test1_dates = result1["test"]["dates"]
    if result2 is not None:
        test2_dates = result2["test"]["dates"]
        test_dates  = test1_dates.append(test2_dates)
    else:
        test_dates  = test1_dates

    train_w      = result1["train"]["w"]
    all_dates    = train_dates.append(test_dates)
    all_weights  = np.vstack([train_w, combined_test_w])

    weights_df = pd.DataFrame(
        all_weights, index=all_dates, columns=assets,
    )
    weights_df.index.name = "date"
    train_weights_df = weights_df.loc[train_dates]
    test_weights_df  = weights_df.loc[test_dates]

    summary = {
        "feature_picks":  feature_picks,
        "feat_names":     feat_names,
        "asset_names":    assets,
        "ppo_params":     ppo_params,
        "dsr_eta":        dsr_eta,
        "lookback":       lookback,
        "total_timesteps": total_timesteps,
        "seed":           seed,
        "do_refit":       do_refit,
        "refit_mid_date": refit_mid_date if do_refit else None,
        "train_period":   [str(dates_index[train_idx[0]].date()),
                           str(dates_index[train_idx[-1]].date())],
        "test_period":    [str(dates_index[test_idx[0]].date()),
                           str(dates_index[test_idx[-1]].date())],
        # Phase 1
        "phase1_test_sharpe":      result1["test"]["sharpe"],
        "phase1_test_ann_return":  result1["test"]["ann_return"],
        "phase1_test_mdd":         result1["test"]["max_drawdown"],
        "phase1_best_es_sharpe":   result1["best_es_sharpe"],
        # Phase 2 (if applicable)
        "phase2_test_sharpe":     result2["test"]["sharpe"] if result2 else None,
        "phase2_test_ann_return": result2["test"]["ann_return"] if result2 else None,
        "phase2_test_mdd":        result2["test"]["max_drawdown"] if result2 else None,
        "phase2_best_es_sharpe":  result2["best_es_sharpe"] if result2 else None,
        # Combined
        "combined_test_sharpe":     annualized_sharpe(combined_test_nets),
        "combined_test_ann_return": annualized_return(combined_test_nets),
        "combined_test_mdd":        max_drawdown(combined_test_nets),
        "ew_test_sharpe":           annualized_sharpe(combined_ew_nets),
        # Weights
        "train_weights":       result1["train"]["w"].tolist(),
        "test_weights":        combined_test_w.tolist(),
        "all_weights":         all_weights,
        # Dated DataFrames
        "weights_df":          weights_df,
        "train_weights_df":    train_weights_df,
        "test_weights_df":     test_weights_df,
    }

    logger.info("=" * 100)
    logger.info("FINAL RUN DONE")
    logger.info(f"combined_test_sharpe={summary['combined_test_sharpe']:.4f}  "
                f"ann_ret={summary['combined_test_ann_return']:.4%}  "
                f"mdd={summary['combined_test_mdd']:.4%}")
    if result2:
        logger.info(f"phase1_sharpe={summary['phase1_test_sharpe']:.4f}  "
                    f"phase2_sharpe={summary['phase2_test_sharpe']:.4f}")

    return summary


---
## 18 · Run Final Train / Test

Execute the full final pipeline using the best hyperparameters recovered in Section 16.

- `do_refit=True` (default) fits a fresh model at mid-test and trades the second half with it.  
- `refit_mid_date` defaults to `"2024-07-01"` (one year into the test window).  
- Set `do_refit=False` to use a single model throughout the test period.


In [ ]:
summary = run_final_train_test(
    df=df,
    assets=assets,
    spreads=spreads,
    feature_picks=feature_picks,
    ppo_params=ppo_params,
    dsr_eta=dsr_eta,
    lookback=lookback,
    out_dir=f"{resultsfolder}/final_run",
    total_timesteps=total_timesteps,
    seed=SEED_PORTFOLIO_DRL_PPO_V9_1_3,
    do_refit=True,
    refit_mid_date="2024-07-01",
)

print(f"Combined test Sharpe : {summary['combined_test_sharpe']:.4f}")
print(f"Combined test Ann Ret: {summary['combined_test_ann_return']:.4%}")
print(f"Combined test MDD    : {summary['combined_test_mdd']:.4%}")


In [ ]:
non_tradable_days = str(PROJECT_ROOT / "01_data/aux/non_tradable_days.txt")
must_have_weights = pd.DataFrame(index=pd.date_range(start="2023-07-01", end="2025-06-30", freq="D"))
non_tradable_dates = pd.to_datetime(
    pd.read_csv(non_tradable_days, header=None)[0]
)
must_have_weights = must_have_weights.drop(non_tradable_dates, errors="ignore")

weights = must_have_weights.join(summary['test_weights_df'], how="left")

weights = weights.fillna(method="ffill").fillna(method="bfill")[assets].values

---
## 19 · Portfolio Metrics

Pass the combined weight matrix to `PortfolioMetrics` for a full performance report and plots.


In [ ]:
import sys
from pathlib import Path
from src.metrics import PortfolioMetrics

pm = PortfolioMetrics()


In [ ]:
pm.summary(weights)


In [ ]:
pm.plot_weights(weights)


In [ ]:
pm.plot_cumulative_returns(weights)


In [ ]:
np.save("DRLPPO_technical_weights.npy", weights)